# Lưu cache UPT hidden-pair11k thành dataset

**Không chạy lại UPT, không train, không cần GPU, Internet OFF.**

Nếu dùng notebook mới: Add Input → Notebook Output → chọn phiên bản đã chạy xong cache11k. Phải có hai file features_rank0.h5/features_rank1.h5; ZIP review không đủ.

Nếu muốn chạy ngay trong notebook cache vừa xong, có thể sao chép hai code cell dưới vào cuối notebook đó.

Notebook kiểm tra đúng cache đã được audit, rồi sao chép sáu file cần thiết sang một thư mục gọn. Không nén HDF5, không cần KAGGLE_API_TOKEN. Không xóa dữ liệu nguồn.


In [ ]:
from pathlib import Path
SOURCE = None  # Tự tìm cache đúng hash trong /kaggle/working hoặc các Input đã thêm.
# Nếu không tìm thấy: SOURCE = Path('/kaggle/input/notebooks/.../upt_hidden_pairs_11k')
DESTINATION = Path('/kaggle/working/upt_hidden_pairs_dataset/upt_hidden_pairs_11k')
print('Packaging verified UPT cache only; no GPU/model/training.', flush=True)


In [ ]:
SOURCE_CODE = '"""Copy a verified hidden-pair cache to a small Kaggle Dataset-ready folder; stdlib only."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\n\nRECEIPT_SHA = \'873f6da0e1b4c4f7d7408b67c593c07ce56e5b4ca3fa1db5f9ce5e04aa188920\'\nREQUIRED = {\'cohort.json\', \'features_rank0.h5\', \'features_rank1.h5\', \'pairs_rank0.jsonl\', \'pairs_rank1.jsonl\'}\n\n\ndef sha(path):\n    result = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 * 1024**2), b\'\'):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef locate(source=None):\n    if source is not None:\n        folder = Path(source)\n        if not (folder / \'cache_receipt.json\').is_file():\n            raise FileNotFoundError(f\'SOURCE must contain cache_receipt.json: {folder}\')\n        return folder\n    local = Path(\'/kaggle/working/upt_hidden_pairs_11k\')\n    if (local / \'cache_receipt.json\').is_file() and sha(local / \'cache_receipt.json\') == RECEIPT_SHA:\n        return local\n    candidates = [p.parent for p in Path(\'/kaggle/input\').rglob(\'cache_receipt.json\') if sha(p) == RECEIPT_SHA]\n    if len(candidates) != 1:\n        raise RuntimeError(f\'Found {len(candidates)} matching cache folders. Attach completed cache notebook Output, then set SOURCE to its exact folder. The review ZIP alone is insufficient.\')\n    return candidates[0]\n\n\ndef package(source, destination, receipt_sha=RECEIPT_SHA):\n    source, destination = Path(source).resolve(), Path(destination).resolve()\n    if source == destination or source in destination.parents or destination in source.parents:\n        raise ValueError(\'Source and destination must be separate, non-nested folders.\')\n    receipt_path = source / \'cache_receipt.json\'\n    if sha(receipt_path) != receipt_sha:\n        raise ValueError(\'Receipt differs from the audited cache; do not silently repackage another run.\')\n    receipt = json.loads(receipt_path.read_text(encoding=\'utf-8\'))\n    if receipt[\'status\'] != \'CACHE_COMPLETE\' or receipt[\'schema\'] != \'upt_hidden_pair_caption_cache_v1\':\n        raise ValueError(\'Incomplete or unknown cache schema.\')\n    if set(receipt[\'files\']) != REQUIRED:\n        raise ValueError(\'Cache payload inventory differs.\')\n    expected = dict(receipt[\'files\'], **{\'cache_receipt.json\': receipt_sha})\n    for name, expected_sha in expected.items():\n        path = source / name\n        if not path.is_file() or sha(path) != expected_sha:\n            raise ValueError(\'Missing or changed source file: \' + name)\n        print(\'Source verified:\', name, flush=True)\n    destination.mkdir(parents=True, exist_ok=True)\n    if any(p.name not in set(expected) | {\'transport_receipt.json\'} | {n + \'.partial\' for n in expected}\n           for p in destination.iterdir()):\n        raise ValueError(\'Destination contains unrelated files; choose a fresh folder.\')\n    needed = sum((source / name).stat().st_size for name in expected if not (destination / name).is_file())\n    if shutil.disk_usage(destination).free < needed + 256 * 1024**2:\n        raise RuntimeError(\'Not enough free disk space for cache copy plus256MiB.\')\n    for name, expected_sha in expected.items():\n        final = destination / name\n        if final.exists():\n            if not final.is_file() or sha(final) != expected_sha:\n                raise ValueError(\'Existing destination differs; preserving it: \' + name)\n            print(\'Already verified:\', name, flush=True)\n            continue\n        temporary = destination / (name + \'.partial\')\n        with (source / name).open(\'rb\') as reader, temporary.open(\'wb\') as writer:\n            shutil.copyfileobj(reader, writer, 8 * 1024**2)\n        if sha(temporary) != expected_sha:\n            raise ValueError(\'Copied bytes differ: \' + name)\n        temporary.replace(final)\n        print(\'Copied and verified:\', name, flush=True)\n    report = dict(status=\'DATASET_FOLDER_READY\', source=str(source), payload_files=6,\n                  files=expected, bytes=sum((destination / n).stat().st_size for n in expected),\n                  model_or_training_executed=False,\n                  next=\'Save notebook Output, then create a private Kaggle Dataset from this output folder. No API token required.\')\n    (destination / \'transport_receipt.json\').write_text(json.dumps(report, indent=2) + \'\\n\', encoding=\'utf-8\')\n    print(\'DATASET FOLDER READY:\', destination, flush=True)\n    print(\'Payload MiB:\', round(report[\'bytes\'] / 1024**2, 2), flush=True)\n    return report\n'
assert __import__("hashlib").sha256(SOURCE_CODE.encode()).hexdigest() == 'f40458c5c996fa88156c5cce0c89a8d0102b3ef181ce0ad32cf06cbed3ef57c8'
exec(compile(SOURCE_CODE, "package_cache_dataset.py", "exec"))
source = locate(SOURCE)
report = package(source, DESTINATION)
print("Save this notebook Output, then create a private dataset from:", DESTINATION, flush=True)


## Sau khi chạy thành công

1. Chọn **Save Version → Save & Run All**; đợi phiên bản hoàn tất để Output được lưu.
2. Vào Output của phiên bản đó, chọn tạo **New Dataset** từ folder `upt_hidden_pairs_dataset`. Nếu Kaggle không hiện lựa chọn này, tải các file trong folder về rồi tạo dataset/upload lại; giữ nguyên file và thư mục.
3. Đặt tên ví dụ **upt-hidden-pairs-caption11k**, để Private. Nếu tài khoản bạn của bạn chạy H, chia sẻ dataset cho tài khoản đó.
4. Trong notebook A/H, sửa `HIDDEN_CACHE` thành đường dẫn thực tế của folder **chứa cache_receipt.json**. Ví dụ `/kaggle/input/datasets/ducanh2403/upt-hidden-pairs-caption11k/upt_hidden_pairs_11k` — số tầng folder có thể khác, hãy copy path từ Input.

Dataset có sáu file payload và một transport_receipt.json. Không cần upload repository, logs hay ZIP review vào dataset training. Không cần tạo ZIP cho từng HDF5.
